In [ ]:
# --- INSTALLATION ---
import os
import sys

print("Installing Research Environment...")

# 1. Clean Slate
!pip uninstall -y jax jaxlib flax tunix qwix libtpu-nightly

# 2. Install JAX (Detects TPU or GPU automatically)
try:
    if os.environ.get("TPU_NAME"):
        !pip install -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
    else:
        !pip install -U "jax[cuda12]"
except:
    pass

# 3. Install Tunix Ecosystem
!pip install git+https://github.com/google/tunix.git
!pip install git+https://github.com/google/qwix.git
!pip install git+https://github.com/google/flax.git


Installing Research Environment...
Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
Found existing installation: flax 0.10.7
Uninstalling flax-0.10.7:
  Successfully uninstalled flax-0.10.7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.1/80.1 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 126.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 769.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

  Cloning https://github.com/google/qwix.git to /tmp/pip-req-build-6v2syfq8
  Running command git clone --filter=blob:none --quiet https://github.com/google/qwix.git /tmp/pip-req-build-6v2syfq8
  Resolved https://github.com/google/qwix.git to commit 98a44ed05b66773724c7daf0f2eb40c9916f6421
ERROR: Operation cancelled by user
^C
  Cloning https://github.com/google/flax.git to /tmp/pip-req-build-c1f45egw
  Running command git clone --filter=blob:none --quiet https://github.com/google/flax.git /tmp/pip-req-build-c1f45egw


In [ ]:
# import sys
# print(sys.version)

In [ ]:
# import jax
# # This is the standard initialization for Colab/Kaggle TPUs
# try:
#     import jax.tools.colab_tpu
#     jax.tools.colab_tpu.setup_tpu()
# except:
#     pass

# print(f"JAX Version: {jax.__version__}")
# # print(f"TPU Devices: {jax.devices()}")

# try:
#     import orbax.checkpoint as ocp
#     # Test if the attribute Tunix wants actually exists
#     policy = ocp.checkpoint_managers.ContinuousCheckpointingPolicy(minimum_interval_secs=180)
#     print("Success: Orbax path is compatible with Tunix!")
# except AttributeError:
#     print("Failure: Orbax version is still too new. Try installing 0.5.0.")
# except Exception as e:
#     print(f"Error: {e}")

# # Now try your Tunix import
# from tunix.generate import tokenizer_adapter as tokenizer_lib
# print("Tunix loaded successfully!")

In [ ]:
!pip install -U huggingface_hub

from huggingface_hub import login
login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.4/533.4 kB 16.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.1 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.3.1 which is incompatible.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Importing Modules

In [ ]:
import os
import re
import json
import logging
from flax import nnx
from huggingface_hub import snapshot_download

import jax
import jax.numpy as jnp

from datasets import load_dataset,concatenate_datasets
import optax
from tunix.generate import tokenizer_adapter as tokenizer_lib

from tunix.models.gemma3 import params_safetensors as params_safetensors_lib
from tunix.models.gemma3 import model as gemma_lib
# from tunix.rl.experimental.agentic_grpo_learner import GRPOConfig, GRPOLearner
from flax import nnx
from tunix.cli.utils import model as model_utils

from orbax import checkpoint as ocp
import qwix

from tunix.sft import metrics_logger
from tunix.sft import peft_trainer
from tunix.sft import utils

logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Gemma 3 1B
model_id = "google/gemma-3-1b-it"


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


## Configurations

In [ ]:
# Configs


# Model Setup
import jax
import os

# Check hardware
NUM_TPUS = len(jax.devices())
print(f"Detected {NUM_TPUS} TPU device(s)")

# CRITICAL: Set these ONCE and NEVER change them
if NUM_TPUS == 1:
    print("Single TPU - Using minimal memory config")
    BATCH_SIZE = 2
    MAX_TARGET_LENGTH = 128
    USE_QUANTIZATION = False
    GRADIENT_ACCUMULATION_STEPS = 32   # Effective batch = 2×32 = 64
    MESH_COUNTS = (1, 1)

elif NUM_TPUS == 8:
    print("Full TPU v2-8 - Using optimized config")
    BATCH_SIZE = 8
    MAX_TARGET_LENGTH = 256
    USE_QUANTIZATION = False           # No QLoRA even with 8 cores
    GRADIENT_ACCUMULATION_STEPS = 8    # Effective batch = 8×8 = 64
    MESH_COUNTS = (1, 4)
else:
    raise ValueError(f"Unsupported: {NUM_TPUS} TPUs")

# Need for GRPO
# TEMPERATURE = 0.9
# TOP_P = 1.0  # implies we don't do nucleus sampling
# TOP_K = 50

# The number of iterations per batch (𝜇 in GRPO algo 1).
# NUM_ITERATIONS = 1
# EPSILON = 0.2


# LoRA/QLoRA Configuration
# Common settings
MESH = [MESH_COUNTS, ("fsdp", "tp")]
RANK = 16
ALPHA = 2.0
MAX_STEPS = 500
EVAL_EVERY_N_STEPS = 50
NUM_EPOCHS = 3

# Checkpoint dir
LORA_CKPT_DIR = "/tmp/content/lora_ckpts/"
os.makedirs(LORA_CKPT_DIR, exist_ok=True)


# Print config
print(f"\nTraining Configuration:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Max length: {MAX_TARGET_LENGTH}")
print(f"   Gradient accum: {GRADIENT_ACCUMULATION_STEPS}")
print(f"   Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"   Quantization: {USE_QUANTIZATION}")
print(f"   Total steps: {MAX_STEPS}")


# Save for later reference
checkpoint_metadata = {
    "base_model": "google/gemma-3-1b-it",
    "lora_rank": RANK,
    "lora_alpha": ALPHA,
    "quantization": USE_QUANTIZATION,
    "max_seq_length": MAX_TARGET_LENGTH,
    "batch_size": BATCH_SIZE,
    "output_format": "<reasoning>...</reasoning><answer>...</answer>"
}

ERROR:2026-01-10 07:52:47,292:jax._src.xla_bridge:475: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/xla_bridge.py", line 473, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/usr/local/lib/python3.12/dist-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/usr/local/lib/python3.12/dist-packages/jax_plugins/xla_cuda12/__init__.py", line 285, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: Unknown CUDA error 303; cuGetErrorName failed. This probably means that JAX was unable to load the CUDA libraries.
ERROR:jax._src.xla_bridge:Jax plugin configuration error: Exception when calling jax_plu

Detected 1 TPU device(s)
Single TPU - Using minimal memory config

Training Configuration:
   Batch size: 2
   Max length: 128
   Gradient accum: 32
   Effective batch: 64
   Quantization: False
   Total steps: 500


## BASE MODEL LOADING...

In [ ]:

ignore_patterns = [
    "*.pth",  # Ignore PyTorch .pth weight files
]
print(f"Downloading {model_id} from Hugging Face...")
local_model_path = snapshot_download(
    repo_id=model_id, ignore_patterns=ignore_patterns
)
print(f"Model successfully downloaded to: {local_model_path}")

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-1b-it/revision/main "HTTP/1.1 200 OK"


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-3-1b-it/resolve/dcc83ea841ab6100d6b47a070329e1ba4cf78752/.gitattributes "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-3-1b-it/resolve/dcc83ea841ab6100d6b47a070329e1ba4cf78752/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-3-1b-it/resolve/dcc83ea841ab6100d6b47a070329e1ba4cf78752/special_tokens_map.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-3-1b-it/resolve/dcc83ea841ab6100d6b47a070329e1ba4cf78752/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-3-1b-it/resolve/dcc83ea841ab6100d6b47a070329e1ba4cf78752/model.safetensors "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-3-1b-it/resolve/dcc83ea841ab6100d6b47a070329e1ba4cf78752/tokenizer.json "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/g

Model successfully downloaded to: /root/.cache/huggingface/hub/models--google--gemma-3-1b-it/snapshots/dcc83ea841ab6100d6b47a070329e1ba4cf78752


In [ ]:
jax.__version__

In [ ]:
import inspect

# List all methods in ModelConfig that start with 'gemma'
configs = [m for m in dir(gemma_lib.ModelConfig) if m.startswith('gemma')]
print("Available Gemma 3 configs:", configs)

# Match the config to the specific variant in your directory
if "gemma-3-1b-it" in model_id:
    model_config = gemma_lib.ModelConfig.gemma3_1b_it()
elif "gemma-3-1b-pt" in model_id:
    model_config = gemma_lib.ModelConfig.gemma3_1b_pt()
elif "gemma-3-1b" in model_id:
    # Default fallback if you're not sure
    print("Defaulting to Pretrained (pt) config...")
    model_config = gemma_lib.ModelConfig.gemma3_1b_pt()
else:
    raise ValueError(f"Unsupported model: {model_id}. Available: {configs}")

Available Gemma 3 configs: ['gemma3_12b_it', 'gemma3_12b_pt', 'gemma3_1b_it', 'gemma3_1b_pt', 'gemma3_270m', 'gemma3_270m_it', 'gemma3_27b_it', 'gemma3_27b_pt', 'gemma3_4b_it', 'gemma3_4b_pt']


In [ ]:
import pprint
from dataclasses import asdict

print("\n Model Configuration Details :) ")
try:
    pprint.pprint(asdict(model_config))
except TypeError:
    # If it's not a standard dataclass, use vars() or dir()
    attrs = {k: getattr(model_config, k) for k in dir(model_config) if not k.startswith('_') and not callable(getattr(model_config, k))}
    pprint.pprint(attrs)


 Model Configuration Details :) 
{'embed_dim': 1152,
 'global_base_frequency': 1000000,
 'global_scale_factor': 1.0,
 'head_dim': 256,
 'hidden_dim': 6912,
 'local_base_frequency': 10000,
 'local_scale_factor': 1.0,
 'num_embed': 262144,
 'num_heads': 4,
 'num_kv_heads': 1,
 'num_layers': 26,
 'param_dtype': <class 'jax.numpy.bfloat16'>,
 'query_pre_attn_norm': <QueryPreAttentionNormalisation.BY_ONE_OVER_SQRT_HEAD_DIM: 1>,
 'remat_config': <RematConfig.NONE: 1>,
 'shd_config': {'act_btd': ('fsdp', None, 'tp'),
                'act_btf': ('fsdp', None, 'tp'),
                'act_btnh': ('fsdp', None, 'tp', None),
                'emb_vd': ('tp', 'fsdp'),
                'ffw_weight_df': ('fsdp', 'tp'),
                'ffw_weight_fd': ('tp', 'fsdp'),
                'kv_weight_cndh': (None, 'tp', 'fsdp', None),
                'o_weight_nhd': ('tp', None, 'fsdp'),
                'q_weight_ndh': ('tp', 'fsdp', None),
                'qkv_weight_cndh': (None, 'tp', 'fsdp', None),
     

In [ ]:
MODEL_CP_PATH = local_model_path

mesh = jax.make_mesh(*MESH, axis_types=(jax.sharding.AxisType.Auto,) * len(MESH[0]))

with mesh:
  base_model = params_safetensors_lib.create_model_from_safe_tensors(
      MODEL_CP_PATH, (model_config), mesh
  )
  # nnx.display(base_model) just calls cluster output

## MODEL TOKENIZATION

In [ ]:
# initialize tokenizer
TOKENIZER_LOCAL_PATH = "gemma-data/tokenizers/tokenizer_gemma3.model"

GEMMA_TOKENIZER_PATH = os.path.join(local_model_path, "tokenizer.model")

if not os.path.exists(GEMMA_TOKENIZER_PATH):
  print(f"File not found at {GEMMA_TOKENIZER_PATH}. Check the directory content:")
  print(os.listdir(local_model_path))

else:
  tokenizer = tokenizer_lib.Tokenizer(tokenizer_path=GEMMA_TOKENIZER_PATH)
  EOS_TOKENS = [tokenizer.eos_id()]
  print(f"Tokenizer loaded. EOS token IDs: {EOS_TOKENS}")


Tokenizer loaded. EOS token IDs: [1]


In [ ]:

# def create_dir(path):
#   try:
#     os.makedirs(path, exist_ok=True)
#     logging.info(f"Created dir: {path}")
#   except OSError as e:
#     logging.error(f"Error creating directory '{path}': {e}")


# # create_dir(FULL_CKPT_DIR)
# create_dir(LORA_CKPT_DIR)


## SYSTEM PROMPT

In [ ]:
SYSTEM_PROMPT = """You are a helpful AI assistant that thinks step by step.
When answering questions:
1. First, provide your reasoning inside <reasoning></reasoning> tags
2. Then, provide your final answer inside <answer></answer> tags"""


## LoRA TRAINER

In [ ]:
def use_lora(base_model, mesh, quantize: bool = False):
    """
    Apply LoRA or QLoRA to base model

    Args:
        base_model: Base Gemma3 model
        mesh: JAX mesh for distributed training
        quantize: If True, use QLoRA with NF4 quantization

    Returns:
        LoRA-adapted model
    """
    # Configure LoRA provider
    if quantize:
        print("\nConfiguring QLoRA with NF4 quantization...")
        lora_provider = qwix.LoraProvider(
            module_path=".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj",
            rank=RANK,
            alpha=ALPHA,
            weight_qtype="nf4",
            tile_size=128,
        )
    else:
        print("\nConfiguring LoRA...")
        lora_provider = qwix.LoraProvider(
            module_path=".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj",
            rank=RANK,
            alpha=ALPHA,
        )

    print("\nGetting model input for tracing...")
    model_input = base_model.get_model_input()

    # Apply LoRA with the model inputs
    print("\nApplying LoRA to model...")
    lora_model = qwix.apply_lora_to_model(
        base_model,
        lora_provider,
        **model_input  # Unpack the input dict (last_tokens, positions, cache, attention_mask)
    )

    # Shard the model across TPU devices
    print("\nSharding model across devices...")
    with mesh:
        state = nnx.state(lora_model)
        pspecs = nnx.get_partition_spec(state)
        sharded_state = jax.lax.with_sharding_constraint(state, pspecs)
        nnx.update(lora_model, sharded_state)

    print("\nLoRA application complete!")
    return lora_model

In [ ]:
# Create LoRA or QLoRA model based on USE_QUANTIZATION hyperparameter
lora_model = use_lora(base_model, mesh=mesh, quantize=USE_QUANTIZATION)

print(f"Using {'QLoRA' if USE_QUANTIZATION else 'LoRA'} model")


Configuring LoRA...

Getting model input for tracing...

Applying LoRA to model...


/usr/local/lib/python3.12/dist-packages/qwix/_src/providers/lora.py:43: UserWarning: rngs must be provided for NNX models to initialize LoRA weights. Please specify rngs=nnx.Rngs(...) in apply_lora_to_model.
  warnings.warn(
INFO:absl:[QWIX] module='layers/0/attn/q_einsum' op=einsum0 rule=0
INFO:absl:[QWIX] module='layers/0/attn/kv_einsum' op=einsum0 rule=0
INFO:absl:[QWIX] module='layers/0/attn' op=einsum0 rule=None
INFO:absl:[QWIX] module='layers/0/attn' op=einsum1 rule=None
INFO:absl:[QWIX] module='layers/0/attn/attn_vec_einsum' op=einsum0 rule=None
INFO:absl:[QWIX] module='layers/0/mlp/gate_proj' op=dot_general0 rule=0
INFO:absl:[QWIX] module='layers/0/mlp/up_proj' op=dot_general0 rule=0
INFO:absl:[QWIX] module='layers/0/mlp/down_proj' op=dot_general0 rule=0
INFO:absl:[QWIX] module='layers/1/attn/q_einsum' op=einsum0 rule=0
INFO:absl:[QWIX] module='layers/1/attn/kv_einsum' op=einsum0 rule=0
INFO:absl:[QWIX] module='layers/1/attn' op=einsum0 rule=None
INFO:absl:[QWIX] module='layers


Sharding model across devices...

LoRA application complete!
Using LoRA model


## RESPONSE EXTRACTION

In [ ]:
def extract_final_number(answer_text):
    """
    Extract the final numerical answer from GSM8K answer format
    GSM8K answers end with "#### NUMBER"
    """
    import re

    # GSM8K format: "reasoning steps\n#### 42"
    match = re.search(r'####\s*([0-9,\.]+)', answer_text)
    if match:
        return match.group(1).replace(',', '')  # Remove commas

    # Fallback: try to find last number
    numbers = re.findall(r'-?\d+\.?\d*', answer_text)
    if numbers:
        return numbers[-1]

    return "0"  # Fallback if no number found

In [ ]:
def extract_answer(answer_text, domain:str="math"):
    """Extract answer based on domain"""
    if domain == "math":
        return extract_final_number(answer_text)
    elif domain == "coding":
        # Extract code block
        return answer_text  # Adjust as needed
    else:
        # For creative/summarization, use full text
        return answer_text

## CHAT TEMPLATE

In [ ]:
def chat_template(raw_text):
    """Add reasoning format to responses"""
    prompt = raw_text['prompt']
    response = raw_text['response']
    domain = raw_text.get('domain', 'general')

    # Create reasoning wrapper
    # For simplicity, use entire response as reasoning
    # and extract key part as answer

    # Extract answer based on domain
    if domain == 'math':
        # Try to get numerical answer
        answer = extract_final_number(response)
    elif domain == 'code':
      code_blocks = re.findall(r'```.*?```', response, re.DOTALL)
      if code_blocks:
          answer = code_blocks[-1]  # Last code block
      else:
          answer = response[:200]  # First 200 chars
    else:
        # For text tasks, use last sentence or summary
        sentences = response.split('.')
        answer = sentences[-1].strip() if sentences else response[:100]

    formatted_response = f"""<reasoning>
{response}
</reasoning>
<answer>
{answer}
</answer>"""

    messages = [
        {
            "role": "user",
            "content": prompt
        },
        {
            "role": "assistant",
            "content": formatted_response
        }
    ]

    return {"messages": messages}

## GENERATION OF ATTENTION AND PADDING TOKENS

In [ ]:
import numpy as np

def gen_model_input_fn(batch):
    """
    Takes pre-tokenized input_ids and creates padded batches with attention masks

    Args:
        batch: Dict with 'input_ids' key containing list of token id sequences
    Returns:
        Dict with 'input_ids' and 'attention_mask' as JAX arrays
    """

    FIXED_LEN = MAX_TARGET_LENGTH

    # Get pre-tokenized sequences
    if isinstance(batch, list):
      token_sequences = [ex['input_ids'] for ex in batch]
    elif isinstance(batch, dict):
      token_sequences = batch['input_ids']
    else:
        raise ValueError(f"Unexpected batch format: {type(batch)}")

    # Handle different input formats (list of lists or 2D array)
    if isinstance(token_sequences, np.ndarray):
        if token_sequences.ndim == 0:
            # Scalar → wrap
            token_sequences = [[token_sequences]]
        elif token_sequences.ndim == 1:
            # Single sequence, wrap in list
            token_sequences = [token_sequences.tolist()]
        else:
            # 2D array, convert to list of lists
            token_sequences = [seq.tolist() for seq in token_sequences]

    # if token_sequence is 0-dimension
    if isinstance(token_sequences,(list,tuple)) and len(token_sequences) ==0:
        token_sequences = [[]]


    # Find max length in this batch
    # max_len = max(len(seq) for seq in token_sequences)

    # Pad sequences and create attention masks
    padded_tokens = []
    attention_masks = []

    for token_ids in token_sequences:
        # Ensure it's a list
        if isinstance(token_ids, np.ndarray):
            token_ids = token_ids.tolist()

        if not isinstance(token_ids, list):
            token_ids = [token_ids]

        token_ids = token_ids[:FIXED_LEN]
        # Calculate padding
        # padding_length = max_len - len(token_ids)
        padding_length = FIXED_LEN - len(token_ids)

        # Pad with pad token
        padded = token_ids + [tokenizer.pad_id()] * padding_length
        mask = [1] * len(token_ids) + [0] * padding_length

        padded_tokens.append(padded)
        attention_masks.append(mask)

    input_tokens = jnp.array(padded_tokens, dtype=jnp.int32)
    input_mask = jnp.array(attention_masks, dtype=jnp.int32)

    positions = jnp.broadcast_to(
        jnp.arange(FIXED_LEN, dtype=jnp.int32),
        input_tokens.shape,
    )

    # Cauing 4D ERROR Broadcasting worked earlier,AND einsum now fails with BTNS
    # Attention mask: reuse padding mask (trainer applies causality internally)
    attention_mask = input_mask[:, None,:]

    return {
        "input_tokens": input_tokens,
        "input_mask": input_mask,
        "positions": positions,
        "attention_mask": attention_mask,
    }

## REBALANCING DATASET

In [ ]:
def prepare_datasets_with_chat_template():
    """
    Load and REBALANCE multi-domain dataset
    """
    # Load full dataset
    ds = load_dataset("Addyk24/Multi-domain-reasoning")

    SAMPLE_SIZES = {
        "math": 2000,
        "coding": 200,
        "science": 3000,
        "general": 3000,
        "summarization": 2000,
        "creative": 1000,
    }

    # Rebalance training set
    print("Rebalancing dataset...")
    train_balanced = []
    val_balanced = []

    for domain, max_size in SAMPLE_SIZES.items():
        # Filter by domain
        domain_data = ds['train'].filter(lambda x: x['domain'] == domain)

        if max_size and len(domain_data) > max_size:
            # Sample down
            domain_data = domain_data.shuffle(seed=42).select(range(max_size))
            print(f"{domain}: {len(domain_data)} (sampled from more)")
        else:
            # Keep all
            print(f"{domain}: {len(domain_data)} (kept all)")

        train_balanced.append(domain_data)

    # Also balance validation set proportionally
    for domain in SAMPLE_SIZES.keys():
        domain_val = ds['validation'].filter(lambda x: x['domain'] == domain)
        if len(domain_val) > 0:
            # Take up to 10% for validation or max 1000 per domain
            val_size = min(len(domain_val), 1000)
            domain_val = domain_val.shuffle(seed=42).select(range(val_size))
            val_balanced.append(domain_val)

    # Concatenate and shuffle
    from datasets import concatenate_datasets
    train_ds = concatenate_datasets(train_balanced).shuffle(seed=42)
    val_ds = concatenate_datasets(val_balanced).shuffle(seed=42)

    print(f"\Balanced dataset:")
    print(f"   Training: {len(train_ds)} examples")
    print(f"   Validation: {len(val_ds)} examples")

    # Format to message structure
    print("\nFormatting messages...")
    train_ds = train_ds.map(
        chat_template,
        remove_columns=['prompt', 'response', 'domain']
    )
    val_ds = val_ds.map(
        chat_template,
        remove_columns=['prompt', 'response', 'domain']
    )
    # only -> ['messages']

    def apply_template_and_tokenize(example):
        """Apply chat template and tokenize in one step"""
        formatted_text = tokenizer.apply_chat_template(
            example['messages'],
            add_generation_prompt=False,
            tokenize=False  # Get string first
        )

        # Tokenize the formatted text
        token_ids = tokenizer.encode(formatted_text)

        # Truncate if needed
        if len(token_ids) > MAX_TARGET_LENGTH:
            token_ids = token_ids[:MAX_TARGET_LENGTH]

        return {"input_ids": token_ids}

    print("Tokenizing and cleaning datasets for JAX/TPU...")

    train_ds = train_ds.map(
        apply_template_and_tokenize,
        remove_columns=['messages']
    )

    val_ds = val_ds.map(
        apply_template_and_tokenize,
        remove_columns=['messages']
    )

    print("Setting numpy format for JAX...")
    train_ds.set_format(type="numpy", columns=["input_ids"])
    val_ds.set_format(type="numpy", columns=["input_ids"])

    print(f"\nFinal datasets:")
    print(f"   Training: {len(train_ds)} examples")
    print(f"   Validation: {len(val_ds)} examples")
    print(f"   Columns: {train_ds.column_names}")

    print("\nExample (decoded):")
    example_ids = train_ds[0]['input_ids']
    decoded = tokenizer.decode(example_ids[:300].tolist() if hasattr(example_ids, 'tolist') else example_ids[:300])
    print(decoded + "...")
    # Should be a 1D array/list of integers
    print(f"Typee: {type(train_ds[0]['input_ids'])}")
    print(f"Shapee: {np.shapee(train_ds[0]['input_ids'])}")

    return train_ds, val_ds

<>:50: SyntaxWarning: invalid escape sequence '\B'
<>:50: SyntaxWarning: invalid escape sequence '\B'
/tmp/ipython-input-3506896209.py:50: SyntaxWarning: invalid escape sequence '\B'
  print(f"\Balanced dataset:")


In [ ]:
import jax
print("="*60)
print("HARDWARE CHECK")
print("="*60)
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Device type: {jax.devices()[0].platform}")
print(f"Number of devices: {len(jax.devices())}")
print("="*60)

HARDWARE CHECK
JAX version: 0.8.2
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Device type: tpu
Number of devices: 1


In [ ]:
# Getting train, validation dataset
train_ds, validation_ds = prepare_datasets_with_chat_template()

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Addyk24/Multi-domain-reasoning/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Addyk24/Multi-domain-reasoning/f074376943728cb778c59bca3a621e4b775d8565/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Addyk24/Multi-domain-reasoning/resolve/f074376943728cb778c59bca3a621e4b775d8565/Multi-domain-reasoning.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Addyk24/Multi-domain-reasoning/Addyk24/Multi-domain-reasoning.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Addyk24/Multi-domain-reasoning/resolve/f074376943728cb778c59bca3a621e4b775d8565/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=Addyk24/Multi-domain-rea

Rebalancing dataset...
math: 2000 (sampled from more)
coding: 200 (sampled from more)
science: 3000 (sampled from more)
general: 3000 (sampled from more)
summarization: 2000 (sampled from more)
creative: 1000 (sampled from more)
\Balanced dataset:
   Training: 11200 examples
   Validation: 4600 examples

Formatting messages...
Tokenizing and cleaning datasets for JAX/TPU...


Map:   0%|          | 0/11200 [00:00<?, ? examples/s]

Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Setting numpy format for JAX...

Final datasets:
   Training: 11200 examples
   Validation: 4600 examples
   Columns: ['input_ids']

Example (decoded):
<start_of_turn>user
List five Italian cheeses.<end_of_turn>
...
Typee: <class 'numpy.ndarray'>


AttributeError: module 'numpy' has no attribute 'shapee'

In [ ]:
print('Input_Ids: ' ,train_ds.column_names)

In [ ]:
print(train_ds[0])

{'input_ids': array([     2,    105,   2364,    107,   1613,   3493,  12896, 104225,
       236761,    106,    107,      1])}


In [ ]:
sample_batch = train_ds[:2]
processed = gen_model_input_fn(sample_batch)

for key, val in processed.items():
    print(f"{key}: {val.shape}")

# TARGET OUTPUT:
# input_tokens: (2, 1024)
# input_mask: (2, 1024)
# positions: (2, 1024)
# attention_mask: (2, 1, 1, 1024)

## POST TRAINING BASE MODEL - Gemma-3n-1B

In [ ]:

lora_logging_options = metrics_logger.MetricsLoggerOptions(
    log_dir="/tmp/tensorboard/lora", flush_every_n_steps=20
)

training_config = peft_trainer.TrainingConfig(
    eval_every_n_steps=EVAL_EVERY_N_STEPS,
    max_steps=MAX_STEPS,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    metrics_logging_options=lora_logging_options,
    checkpoint_root_directory=LORA_CKPT_DIR,
)

trainer = peft_trainer.PeftTrainer(
    lora_model, optax.adamw(1e-3), training_config
).with_gen_model_input_fn(gen_model_input_fn)

# The first couple of training step might take up to 5 minutes to finish. Please be patient. If you experience long training steps, e.g. >10 minutes per step, please open a bug. Really appreciated!
method_name = "QLoRA" if USE_QUANTIZATION else "LoRA"
with mesh:
    trainer.train(train_ds, validation_ds)


INFO:absl:save_device_host_concurrent_bytes=None
INFO:absl:Created BasePyTreeCheckpointHandler: use_ocdbt=True, use_zarr3=False, pytree_metadata_options=PyTreeMetadataOptions(support_rich_types=False), array_metadata_store=<orbax.checkpoint._src.metadata.array_metadata_store.Store object at 0x7ce068c35cd0>, enable_pinned_host_transfer=False, save_concurrent_bytes: 96000000000 (89.4 GiB), restore_concurrent_bytes: 96000000000 (89.4 GiB)
INFO:absl:save_device_host_concurrent_bytes=None
INFO:absl:Created BasePyTreeCheckpointHandler: use_ocdbt=True, use_zarr3=False, pytree_metadata_options=PyTreeMetadataOptions(support_rich_types=False), array_metadata_store=<orbax.checkpoint._src.metadata.array_metadata_store.Store object at 0x7ce068c35cd0>, enable_pinned_host_transfer=False, save_concurrent_bytes: 96000000000 (89.4 GiB), restore_concurrent_bytes: 96000000000 (89.4 GiB)
INFO:absl:[process=0][thread=MainThread] CheckpointManager init: checkpointers=None, item_names=None, item_handlers={'mo

In [ ]:
# After training completes
print("\n" + "="*60)
print("CHECKPOINT SAVED")
print("="*60)
print(f"Checkpoint location: {LORA_CKPT_DIR}")
print(f"Base model: {model_id}")
print(f"LoRA config: rank={RANK}, alpha={ALPHA}")
print(f"Quantization: {USE_QUANTIZATION}")
print("\nTo load this checkpoint:")
print("1. Load base model from Hugging Face")
print("2. Apply LoRA with same config")
print("3. Load checkpoint weights from:", LORA_CKPT_DIR)
print("="*60)

# Save metadata
with open(os.path.join(LORA_CKPT_DIR, "training_metadata.json"), "w") as f:
    json.dump(checkpoint_metadata, f, indent=2)

## Inferenece

In [ ]:
def generate_reasoning_response(prompt:str, max_tokens:int=512):
  """
      Generate response with reasoning format
    Args:
        prompt: User query string
        max_tokens: Maximum tokens to generate
    Returns:
        Generated response string
  """

  messages = [{"role": "user", "content": prompt}]

  # Format prompt
  formatted_prompt = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      tokenize=False,
  )

  # Tokenize
  input_ids = tokenizer.encode(formatted_prompt)
  input_len = len(input_ids)
  # Convert to JAX array with proper shape
  input_ids_array = jnp.array([input_ids], dtype=jnp.int32)

  # Generation of outputs
  with mesh:

    output_ids = lora_model.generate(
      input_ids=input_ids_array,
      max_new_tokens=max_tokens,
      temperature=0.7,
      top_p=0.9,
      eos_token_ids=EOS_TOKENS,
    )

    # Decode only the generated portion (skip input)
    generated_ids = output_ids[0][input_len:]  # Remove input tokens
    response = tokenizer.decode(generated_ids.tolist())

    return response


# Test after training
print("\n" + "="*60)
print("TESTING MODEL")
print("="*60)

test_prompts = [
    "What is 25% of 80?",
    "Explain photosynthesis in simple terms",
    "Write a haiku about coding"
]

for test_prompt in test_prompts:
    print(f"\nPrompt: {test_prompt}")
    try:
        response = generate_reasoning_response(test_prompt, max_tokens=512)
        print(f"Response:\n{response}\n")
        print("-" * 60)
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
# Training dataset composition:
# Maths : 	8,790 (Use All)	100%
# Creative : 9510	16%
# Reasoning : 	15,000	28.84%
# Science : 	11,700 (Use All)	100%
# Summarization : 10,000	16.6%
# Coding : 	5,000 (Use All)	100%
# Total : 	50,000	100%

# Validation dataset composition:
# General Reasoning 1,500	30%
# Maths 750	15%
# Science	500	10%
# Coding  500 (All)	100%
# Summarization	750	15%
# Creative	1,000	20%
# TOTAL		5,000	100%